검증 pydantic : 

from pydantic import Field # 추가 제약조건

from pydantic import filed_validater
@field_validater() #특정 필드 제약조건

03. output_parser: LLM의 출력을 더 유용/구조화된 형태로 변환

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """아래 이메일 내용 중 중요한 것만 추출해.
{email_conversation}"""
)

# print(prompt.format(email_conversation=email_conversation))

chain = prompt | llm
chain.invoke({'email_conversaton': email_conversation}).content


In [ ]:
class EmailSummary(BaseModel):
    person: str = Field(description='메일 보낸 사람')
    email: str
    subject: str
    summary: str
    date: str

parser = PydanticOutputParser(pydantic_object=EmailSummary)
print(parser.get_format_instructions())

In [ ]:
prompt = PromptTemplate.from_template(
    """
너는 요약의 신 어시스턴트야. 아래 질문에 맞게 답변을 한국어로 만들어줘

질문: {question}

이메일 내용: {email_conversation}

형식: {format}

"""
)

prompt = prompt.partial(format=parser.get_format_instructions()) 

In [ ]:
chain = prompt | llm | parser

chain.invoke(
    {
        'question': '이메일 내용 중 중요한 내용을 추출해 줘!',
        'email_conversation': email_conversation
    }
)

print(res)

In [ ]:
from pprint import pprint
pprint(res.model_dump())